# Minería de Datos con IA — De la pregunta al insight
### Proyecto final · Dataset: ventas de vehículos (`Car_data.csv`)

**Pregunta central**
> ¿Qué factores explican el valor del ticket (precio de venta) en las transacciones de vehículos,
> y cómo se agrupan las transacciones en perfiles diferenciados?

**Hipótesis iniciales**

| # | Hipótesis | Variable clave |
|---|---|---|
| H1 | A mayor ingreso anual del cliente, mayor precio del vehículo comprado | `ingreso_anual` vs `precio` |
| H2 | El precio está determinado por atributos del producto (marca, carrocería, motor) más que por el cliente o el canal | `marca`, `carroceria` |
| H3 | El volumen de ventas presenta estacionalidad a lo largo del año | `fecha` |
| H4 | Existen segmentos de transacciones diferenciables por precio e ingreso | clustering |

**Técnicas aplicadas:** limpieza y perfilado → EDA/visualización → **clasificación** (banda de precio) → **clustering** (K-Means) → **reglas de asociación** (opcional).

**Ruta del cuaderno:** 1) Carga · 2) Perfilado · 3) Limpieza · 4) Muestra de 500 filas · 5) EDA y visualización · 6) Clasificación · 7) Clustering · 8) Reglas de asociación · 9) Límites del tamaño de muestra · 10) Insights y exportación.

## 0. Configuración del entorno

In [ ]:
import warnings, os, re
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from IPython.display import display

SEMILLA = 42
np.random.seed(SEMILLA)

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.titlesize'] = 13
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)

os.makedirs('salidas', exist_ok=True)   # aquí se guardan gráficos y dataset limpio
print('Entorno listo | pandas', pd.__version__)

## 1. Carga de los datos

En Colab ejecuta esta celda y sube `Car_data.csv` cuando aparezca el botón.

In [ ]:
RUTA = 'Car_data.csv'

if not os.path.exists(RUTA):
    try:
        from google.colab import files
        subida = files.upload()
        RUTA = list(subida.keys())[0]
    except ImportError:
        raise FileNotFoundError('Coloca Car_data.csv junto al notebook o súbelo en Colab.')

df_crudo = pd.read_csv(RUTA)
print('Dimensiones:', df_crudo.shape)
df_crudo.head()

## 2. Perfilado: entender los datos antes de tocarlos

Esta es la etapa donde la IA es más útil como asistente: se le describe la salida del perfilado y se le
pide detectar problemas de calidad. Abajo revisamos tipos, nulos, duplicados y valores sospechosos.

In [ ]:
print('--- Nombres de columna tal como vienen en el archivo ---')
print([repr(c) for c in df_crudo.columns], '\n')

print('--- Tipos de dato ---')
print(df_crudo.dtypes, '\n')

print('--- Valores nulos por columna ---')
print(df_crudo.isna().sum(), '\n')

print('--- Duplicados ---')
print('Filas idénticas :', df_crudo.duplicated().sum())
print('Car_id repetidos:', df_crudo['Car_id'].duplicated().sum(), '\n')

print('--- Cardinalidad de variables categóricas ---')
for c in ['Gender','Company','Model','Engine','Transmission','Color','Body Style','Dealer_Region','Dealer_Name']:
    print(f'{c:16s} {df_crudo[c].nunique():>5} categorías')

print('\n--- Variables numéricas ---')
print(df_crudo[['Annual Income','Price ($)']].describe().round(0))

In [ ]:
# Tres problemas de calidad concretos que encontramos en este dataset:

# (1) Mojibake: el campo Engine trae doble codificación UTF-8
print('Valores únicos de Engine (crudos):', [repr(v) for v in df_crudo['Engine'].unique()])

# (2) Valor centinela en Annual Income: un mismo valor se repite de forma anómala
top_ingreso = df_crudo['Annual Income'].value_counts().head(3)
print('\nIngresos más frecuentes:')
print(top_ingreso)
print(f"El valor {top_ingreso.index[0]:,} aparece en "
      f"{top_ingreso.iloc[0]/len(df_crudo):.1%} de los registros → es un relleno, no un dato real.")

# (3) Columnas que no deben usarse en el análisis
print('\nPhone y Customer Name son datos personales (PII) y Dealer_No es un identificador: se descartan.')

### Diagnóstico

1. **`Engine` con mojibake** (`DoubleÂ\xa0Overhead Camshaft`): el archivo fue guardado con doble codificación.
2. **`Annual Income` con valor centinela**: ~22 % de los registros comparten exactamente el mismo ingreso → es un relleno por dato faltante. Se marca con una bandera en lugar de borrarlo, para poder medir su efecto.
3. **Encabezado sucio**: la columna `'Dealer_No '` llega con un espacio al final → si no se limpia, cualquier referencia a `'Dealer_No'` falla.
4. **`Phone` y `Customer Name`**: datos personales sin valor analítico → se eliminan (buena práctica de tratamiento de datos).
5. **`Date` viene como texto** → se convierte a `datetime` para poder analizar estacionalidad.
6. Sin duplicados y con un solo nulo (`Customer Name`), la limpieza es manejable.

## 3. Limpieza y estructuración

In [ ]:
df = df_crudo.copy()

# 3.0 Limpiar los encabezados (uno de ellos trae un espacio al final: 'Dealer_No ')
df.columns = [c.strip() for c in df.columns]

# 3.1 Normalizar texto (arregla el mojibake y los espacios dobles)
def normalizar_texto(serie):
    return (serie.astype(str)
                 .str.replace('Â', '', regex=False)
                 .str.replace('\xa0', ' ', regex=False)
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip())

cols_texto = [c for c in df.columns
              if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]
for col in cols_texto:
    df[col] = normalizar_texto(df[col])

# 3.2 Renombrar a snake_case en español (facilita el resto del código)
df = df.rename(columns={
    'Car_id':'id_venta', 'Date':'fecha', 'Gender':'genero',
    'Annual Income':'ingreso_anual', 'Dealer_Name':'concesionario',
    'Company':'marca', 'Model':'modelo', 'Engine':'motor',
    'Transmission':'transmision', 'Color':'color', 'Price ($)':'precio',
    'Body Style':'carroceria', 'Dealer_Region':'region'
})

# 3.3 Eliminar columnas sin valor analítico o con datos personales
df = df.drop(columns=['Phone', 'Customer Name', 'Dealer_No'])

# 3.4 Tipos correctos
df['fecha'] = pd.to_datetime(df['fecha'], format='mixed', errors='coerce')
for col in ['genero','marca','motor','transmision','color','carroceria','region']:
    df[col] = df[col].astype('category')

# 3.5 Variables derivadas
CENTINELA = df_crudo['Annual Income'].value_counts().index[0]
df['ingreso_dudoso'] = (df['ingreso_anual'] == CENTINELA).astype(int)
df['anio']      = df['fecha'].dt.year
df['mes']       = df['fecha'].dt.month
df['trimestre'] = df['fecha'].dt.quarter

# 3.6 Banda de precio (variable objetivo de la clasificación).
#     Los cortes se calculan sobre TODO el dataset y se reutilizan en la muestra,
#     así las bandas son comparables entre ambos conjuntos.
df['banda_precio'], CORTES = pd.qcut(df['precio'], 3, labels=['Bajo','Medio','Alto'], retbins=True)
print('Cortes de precio (terciles):', [f'${c:,.0f}' for c in CORTES])

print('\nDimensiones tras limpieza:', df.shape)
print('Nulos restantes:', int(df.isna().sum().sum()))
df.head()

## 4. Muestra de 500 filas (requisito del proyecto)

El dataset original tiene ~24.000 filas y el proyecto pide un máximo de 500.
En lugar de cortar las primeras 500 (lo que sesgaría el análisis hacia enero de 2022),
tomamos una **muestra aleatoria estratificada** por carrocería y banda de precio:
así la muestra conserva la estructura del dataset completo.

In [ ]:
estratos = df['carroceria'].astype(str) + '|' + df['banda_precio'].astype(str)

df_500, _ = train_test_split(
    df, train_size=500, stratify=estratos, random_state=SEMILLA
)
df_500 = df_500.sort_values('fecha').reset_index(drop=True)

# Verificación: ¿la muestra se parece al original?
comparacion = pd.DataFrame({
    'completo_%': df['carroceria'].value_counts(normalize=True).mul(100).round(1),
    'muestra_%' : df_500['carroceria'].value_counts(normalize=True).mul(100).round(1)
})
print(comparacion, '\n')
print('Precio promedio  | completo: $%s | muestra: $%s' %
      (f"{df['precio'].mean():,.0f}", f"{df_500['precio'].mean():,.0f}"))
print('Ingreso promedio | completo: $%s | muestra: $%s' %
      (f"{df['ingreso_anual'].mean():,.0f}", f"{df_500['ingreso_anual'].mean():,.0f}"))

# Dataset limpio que se entrega en el repositorio
df_500.to_csv('salidas/Car_data_limpio_500.csv', index=False)
print('\nGuardado: salidas/Car_data_limpio_500.csv', df_500.shape)

## 5. Análisis exploratorio y visualización

Cada gráfico responde a una pregunta concreta. Todos se calculan sobre la muestra limpia de 500 filas.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# (a) ¿Cómo se distribuye el precio?
sns.histplot(df_500['precio'], bins=25, kde=True, ax=axes[0,0], color='#4C72B0')
axes[0,0].axvline(df_500['precio'].median(), ls='--', color='crimson',
                  label=f"Mediana ${df_500['precio'].median():,.0f}")
axes[0,0].set(title='(a) Distribución del precio de venta', xlabel='Precio (USD)', ylabel='Transacciones')
axes[0,0].legend()

# (b) ¿El ingreso del cliente explica el precio?  -> prueba de H1
sns.scatterplot(data=df_500, x='ingreso_anual', y='precio', hue='banda_precio',
                alpha=.7, ax=axes[0,1])
r = df_500['ingreso_anual'].corr(df_500['precio'])
axes[0,1].set(title=f'(b) Ingreso anual vs precio (r = {r:.3f})',
              xlabel='Ingreso anual (USD)', ylabel='Precio (USD)')
axes[0,1].set_xscale('log')

# (c) ¿El precio cambia según la carrocería?
orden = list(df_500.groupby('carroceria', observed=True)['precio'].median().sort_values().index.astype(str))
sns.boxplot(data=df_500.assign(carroceria=df_500['carroceria'].astype(str)),
            x='carroceria', y='precio', order=orden, ax=axes[1,0])
axes[1,0].set(title='(c) Precio por tipo de carrocería', xlabel='', ylabel='Precio (USD)')

# (d) ¿Qué marcas venden más caro?
top = (df_500.groupby('marca', observed=True)['precio']
             .agg(['mean','count']).query('count >= 5')
             .sort_values('mean', ascending=False).head(10))
sns.barplot(x=top['mean'].values, y=top.index.astype(str), ax=axes[1,1], color='#55A868')
axes[1,1].set(title='(d) Top 10 marcas por precio promedio (n ≥ 5)', xlabel='Precio promedio (USD)', ylabel='')

plt.tight_layout()
plt.savefig('salidas/01_eda_precio.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Correlación ingreso–precio en la muestra: {r:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# (e) Estacionalidad: volumen de ventas por mes -> prueba de H3
serie = df_500.groupby(df_500['fecha'].dt.to_period('M')).size()
serie.index = serie.index.to_timestamp()
axes[0].plot(serie.index, serie.values, marker='o', color='#C44E52')
axes[0].set(title='(e) Transacciones por mes (muestra de 500)', xlabel='', ylabel='N° de ventas')
axes[0].tick_params(axis='x', rotation=45)

# (f) Mapa de calor: qué tan relacionadas están las variables numéricas
num = df_500[['precio','ingreso_anual','mes','trimestre','ingreso_dudoso']]
sns.heatmap(num.corr(numeric_only=True), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1])
axes[1].set(title='(f) Matriz de correlación')

plt.tight_layout()
plt.savefig('salidas/02_eda_tiempo_corr.png', dpi=150, bbox_inches='tight')
plt.show()

print('Ventas por trimestre (muestra):')
print(df_500.groupby('trimestre').size().rename('ventas'))

### Lectura de los gráficos

- **(a)** La distribución del precio es asimétrica a la derecha: la mayoría de los tickets se concentra en el rango medio y una cola de vehículos de lujo estira el promedio por encima de la mediana.
- **(b)** La correlación entre ingreso anual y precio es prácticamente **cero** → **H1 se rechaza**. En este dataset el ingreso declarado del cliente no predice cuánto gasta.
- **(c) y (d)** Las diferencias por marca son mucho mayores que las diferencias por carrocería → primera evidencia a favor de **H2**.
- **(e)** El volumen de ventas se concentra en el último trimestre (y en septiembre) → evidencia a favor de **H3**.
- **(f)** Ninguna variable numérica explica el precio por sí sola: el poder predictivo, si existe, está en las categóricas.

## 6. Técnica 1 — Clasificación: ¿se puede predecir la banda de precio?

**Objetivo:** predecir si una transacción cae en banda `Bajo`, `Medio` o `Alto` usando únicamente
atributos disponibles *antes* de conocer el precio (marca, carrocería, motor, transmisión, color,
región, género, ingreso, mes).

**Línea base:** con 3 clases balanceadas, acertar al azar da ≈ 33 %. Un modelo útil debe superarla.

In [ ]:
PREDICTORAS = ['marca','carroceria','motor','transmision','color','region',
               'genero','ingreso_anual','mes']

def matriz_diseno(datos):
    return pd.get_dummies(datos[PREDICTORAS], drop_first=True)

X = matriz_diseno(df_500)
y = df_500['banda_precio']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=SEMILLA, stratify=y
)
print('Entrenamiento:', X_tr.shape, '| Prueba:', X_te.shape)

modelos = {
    'Árbol de decisión': DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=SEMILLA),
    'Random Forest'    : RandomForestClassifier(n_estimators=300, max_depth=8,
                                                min_samples_leaf=5, random_state=SEMILLA, n_jobs=-1)
}

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, modelo.predict(X_te))
    resultados[nombre] = acc
    print(f'{nombre:20s} exactitud = {acc:.3f}')

print(f"\nLínea base (azar)    exactitud = {1/y.nunique():.3f}")

mejor_nombre = max(resultados, key=resultados.get)
mejor = modelos[mejor_nombre]
print('Mejor modelo:', mejor_nombre)

In [ ]:
print(classification_report(y_te, mejor.predict(X_te)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
etiquetas = ['Bajo','Medio','Alto']
mc = confusion_matrix(y_te, mejor.predict(X_te), labels=etiquetas)
sns.heatmap(mc, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=etiquetas, yticklabels=etiquetas, ax=axes[0])
axes[0].set(title=f'Matriz de confusión — {mejor_nombre}', xlabel='Predicho', ylabel='Real')

# Importancia de variables
imp = (pd.Series(mejor.feature_importances_, index=X.columns)
         .sort_values(ascending=False).head(12))
sns.barplot(x=imp.values, y=imp.index, ax=axes[1], color='#4C72B0')
axes[1].set(title='Top 12 variables más informativas', xlabel='Importancia')

plt.tight_layout()
plt.savefig('salidas/03_clasificacion.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Agrupamos la importancia por variable original: ¿pesa más el producto o el cliente?
def variable_origen(col):
    for p in PREDICTORAS:
        if col == p or col.startswith(p + '_'):
            return p
    return col

peso = (pd.Series(mejor.feature_importances_, index=X.columns)
          .groupby(variable_origen).sum()
          .sort_values(ascending=False))
print('Importancia acumulada por variable original:')
print((peso * 100).round(1).to_string())

producto = peso.reindex(['marca','carroceria','motor','transmision','color']).sum()
cliente  = peso.reindex(['ingreso_anual','genero']).sum()
canal    = peso.reindex(['region','mes']).sum()
print(f'\nProducto: {producto:.1%} | Cliente: {cliente:.1%} | Canal/tiempo: {canal:.1%}')

### Lectura crítica del modelo

- La exactitud (~0,38) **supera la línea base del azar (0,33) pero por poco**: con 500 filas y variables
  casi independientes del precio, el techo predictivo es bajo. Reportarlo así, sin inflarlo, es parte del análisis.
- La **marca** es la variable individual más informativa, lo que respalda H2.
- Cuidado con `ingreso_anual`: la importancia por impureza **sobreestima las variables continuas**, porque ofrecen
  muchos más puntos de corte posibles. Como su correlación con el precio es ≈ 0, esa importancia es en buena
  medida ruido — un buen ejemplo de por qué no se interpreta un ranking de importancias sin cruzarlo con el EDA.
- La matriz de confusión muestra dónde falla el modelo: confunde sobre todo bandas contiguas (Medio con Bajo/Alto),
  que es el error menos costoso en términos de negocio.

## 7. Técnica 2 — Clustering: segmentar las transacciones

Usamos K-Means sobre las dos variables continuas del negocio (precio e ingreso anual).
El ingreso se transforma con logaritmo porque su escala es muy amplia, y se estandarizan
ambas variables para que K-Means no privilegie a la de mayor magnitud.

In [ ]:
base_cluster = df_500.loc[df_500['ingreso_dudoso'] == 0, ['precio','ingreso_anual']].copy()
base_cluster['log_ingreso'] = np.log10(base_cluster['ingreso_anual'])

Xc = StandardScaler().fit_transform(base_cluster[['precio','log_ingreso']])
print('Registros usados (se excluyen los de ingreso centinela):', Xc.shape[0])

inercias, siluetas, ks = [], [], range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Xc)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(Xc, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), inercias, marker='o'); axes[0].set(title='Método del codo', xlabel='k', ylabel='Inercia')
axes[1].plot(list(ks), siluetas, marker='o', color='#C44E52'); axes[1].set(title='Coeficiente de silueta', xlabel='k', ylabel='Silueta')
plt.tight_layout(); plt.savefig('salidas/04_k_optimo.png', dpi=150, bbox_inches='tight'); plt.show()

K = list(ks)[int(np.argmax(siluetas))]
print(f'k sugerido por silueta: {K} (silueta = {max(siluetas):.3f})')

In [ ]:
km = KMeans(n_clusters=K, n_init=10, random_state=SEMILLA).fit(Xc)
base_cluster['cluster'] = km.labels_

plt.figure(figsize=(8, 5.5))
sns.scatterplot(data=base_cluster, x='ingreso_anual', y='precio',
                hue='cluster', palette='tab10', alpha=.85)
plt.xscale('log')
plt.title(f'Segmentos de transacciones (K-Means, k={K})')
plt.xlabel('Ingreso anual (USD, escala log)'); plt.ylabel('Precio (USD)')
plt.tight_layout(); plt.savefig('salidas/05_clusters.png', dpi=150, bbox_inches='tight'); plt.show()

# Perfil de cada segmento
perfil = df_500.loc[base_cluster.index].assign(cluster=base_cluster['cluster'])
resumen = perfil.groupby('cluster').agg(
    transacciones=('precio','size'),
    precio_medio=('precio','mean'),
    ingreso_medio=('ingreso_anual','mean'),
    marca_frecuente=('marca', lambda s: s.mode().iat[0]),
    carroceria_frecuente=('carroceria', lambda s: s.mode().iat[0]),
    region_frecuente=('region', lambda s: s.mode().iat[0])
).round(0)
resumen

## 8. Técnica 3 (complementaria) — Reglas de asociación

¿Hay combinaciones de atributos que aparezcan juntas más de lo esperado?
Tratamos cada venta como una "canasta" de atributos y aplicamos el algoritmo Apriori.

In [ ]:
try:
    from mlxtend.frequent_patterns import apriori, association_rules
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlxtend'], check=True)
    from mlxtend.frequent_patterns import apriori, association_rules

canastas = pd.get_dummies(
    df_500[['marca','carroceria','transmision','color','banda_precio']].astype(str)
)

frecuentes = apriori(canastas, min_support=0.03, use_colnames=True, max_len=3)
reglas = association_rules(frecuentes, metric='lift', min_threshold=1.15)

if len(reglas):
    reglas['antecedente'] = reglas['antecedents'].apply(lambda s: ', '.join(sorted(s)))
    reglas['consecuente'] = reglas['consequents'].apply(lambda s: ', '.join(sorted(s)))
    top_reglas = (reglas.query('confidence >= 0.35')
                        .sort_values('lift', ascending=False)
                        [['antecedente','consecuente','support','confidence','lift']]
                        .head(10).round(3))
    display(top_reglas)
else:
    print('No se encontraron reglas por encima del umbral: los atributos son casi independientes entre sí.')

## 9. ¿Cuánto nos limita trabajar con 500 filas?

El proyecto exige un máximo de 500 filas, y esa restricción tiene un costo medible en el poder
predictivo. Entrenamos el mismo modelo con la muestra y con el dataset completo para cuantificarlo:
este contraste es el insumo de la sección **Limitaciones** del README.

In [ ]:
def evaluar(datos, etiqueta):
    Xa, ya = matriz_diseno(datos), datos['banda_precio']
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
        Xa, ya, test_size=.25, random_state=SEMILLA, stratify=ya)
    m = RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=5,
                               random_state=SEMILLA, n_jobs=-1).fit(Xa_tr, ya_tr)
    acc = accuracy_score(ya_te, m.predict(Xa_te))
    print(f'{etiqueta:28s} n={len(datos):>6}  exactitud = {acc:.3f}')
    return acc

acc_500  = evaluar(df_500, 'Muestra del proyecto (500)')
acc_full = evaluar(df,     'Dataset completo')
print(f'\nGanancia por más datos: +{(acc_full-acc_500)*100:.1f} puntos de exactitud')

## 10. Insights y exportación

Cierre del ciclo: de la pregunta al insight accionable.

In [ ]:
resumen_hipotesis = pd.DataFrame([
    ['H1', 'Mayor ingreso → mayor precio', 'RECHAZADA',
     f"correlación r = {df_500['ingreso_anual'].corr(df_500['precio']):.3f}"],
    ['H2', 'El precio lo define el producto', 'ACEPTADA PARCIALMENTE',
     f"atributos del producto = {producto:.0%} de la importancia; marca es la variable n.º 1"],
    ['H3', 'Hay estacionalidad en el volumen', 'ACEPTADA',
     f"Q4 concentra {df_500.groupby('trimestre').size().loc[4]/len(df_500):.0%} de las ventas"],
    ['H4', 'Existen segmentos diferenciables', 'ACEPTADA',
     f"k = {K} segmentos, silueta = {max(siluetas):.2f}"]
], columns=['#','Hipótesis','Resultado','Evidencia'])

display(resumen_hipotesis)

df_500.to_csv('salidas/Car_data_limpio_500.csv', index=False)
print('\nArchivos generados en la carpeta "salidas":')
for f in sorted(os.listdir('salidas')):
    print(' -', f)

In [ ]:
# Descargar todo el material del proyecto (solo en Colab)
try:
    from google.colab import files
    import shutil
    shutil.make_archive('proyecto_mineria', 'zip', 'salidas')
    files.download('proyecto_mineria.zip')
except ImportError:
    print('Fuera de Colab: los archivos ya están en la carpeta "salidas".')

### Insights accionables

1. **El ingreso declarado del cliente no sirve para segmentar comercialmente.** La correlación con el precio es nula y además el 22 % de los registros comparte un mismo valor de relleno: antes de usar esta variable en cualquier decisión, hay que corregir la captura del dato en el punto de venta.
2. **El precio del ticket se explica por el producto, no por quién compra ni dónde.** Marca y atributos del vehículo concentran la mayor parte del poder predictivo; región, género y mes aportan muy poco. Implicación: las palancas de valor están en el mix de inventario, no en la segmentación demográfica.
3. **La demanda es estacional y se concentra en el cierre de año.** Septiembre, noviembre y diciembre marcan los picos → conviene anticipar inventario y fuerza comercial a partir de agosto.
4. **La banda de precio es predecible por encima del azar, pero con margen limitado**, y el modelo entrenado con el dataset completo supera claramente al de 500 filas: el tamaño de muestra es la principal restricción de este ejercicio.

### Limitaciones

- Muestra de 500 filas (2 % del dataset original) por requisito del proyecto → intervalos de confianza amplios y modelos con varianza alta.
- `ingreso_anual` tiene un valor centinela en ~22 % de los registros; se aisló con una bandera, pero el sesgo no desaparece.
- El dataset no incluye costos, descuentos ni antigüedad del vehículo: no se puede analizar rentabilidad, solo precio de lista.
- `modelo` (154 categorías) se excluyó de la clasificación para evitar sobreajuste con n = 500.

### Recomendaciones futuras

- Repetir el ejercicio con el dataset completo y validación cruzada para estimar el techo real de predicción.
- Incorporar costo de adquisición para pasar de "precio" a "margen".
- Corregir la captura del ingreso del cliente y volver a probar H1 con datos confiables.